<a href="https://colab.research.google.com/github/9terry-student/RWKV/blob/main/RWKV_from_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np

def time_mixing(x, x_prev,num,den):
    # x: 현재 입력 (d_model,)
    # x_prev: 이전 입력 (d_model,)
    # w: 시간 감쇠 파라미터
    # u: 현재 토큰 가중치

    # 1. R, K, V 계산 (현재 + 이전 입력 섞기)
    # R = W_r @ (현재와 이전 입력 믹스)
    W_r=np.random.randn(d_model,d_model)
    u_r=1/(1+np.exp(-np.random.randn(d_model,)))
    R=W_r@(u_r*x+(1-u_r)*x_prev)

    # K = W_k @ (현재와 이전 입력 믹스)
    W_k=np.random.randn(d_model,d_model)
    u_k=1/(1+np.exp(-np.random.randn(d_model,)))
    K=W_k@(u_k*x+(1-u_k)*x_prev)

    # V = W_v @ (현재와 이전 입력 믹스)
    W_v=np.random.randn(d_model,d_model)
    u_v=1/(1+np.exp(-np.random.randn(d_model,)))
    V=W_v@(u_v*x+(1-u_v)*x_prev)

    # 2. WKV 계산 (이전 상태 누적)
    # numerator   = exp(u + k) * v + 이전 누적값
    numerator=np.exp(u_k+K)*V+num

    # denominator = exp(u + k) + 이전 누적 가중치
    denominator=np.exp(u_k+K)+den

    # 3. 출력
    # output = sigmoid(R) * (numerator / denominator)
    output=(numerator/denominator)/(1+np.exp(-R))

    return output,numerator,denominator

def channel_mixing(x, x_prev):
    # x: 현재 입력 (d_model,)
    # x_prev: 이전 입력 (d_model,)

    # 1. R, K 계산 (현재 + 이전 입력 믹스)
    W_r=np.random.randn(d_model,d_model)
    u_r=1/(1+np.exp(-np.random.randn(d_model,)))
    R=W_r@(u_r*x+(1-u_r)*x_prev)
    W_k=np.random.randn(d_model,d_model)
    u_k=1/(1+np.exp(-np.random.randn(d_model,)))
    K=W_k@(u_k*x+(1-u_k)*x_prev)


    # 2. 출력
    # output = sigmoid(R) * ReLU(K)²
    output=(np.maximum(0,K)**2)/(1+np.exp(-R))

    return output

def layernorm(x):
  mean=np.mean(x,axis=-1,keepdims=True)
  std=np.std(x,axis=-1,keepdims=True)
  return (x-mean)/(std+10**(-9))

def rwkv_block(x):
    # x: (seq_len, d_model)
    seq_len,d_model=x.shape

    # 1. 토큰마다 루프
    outputs=[]
    num=np.zeros(d_model)
    den=np.zeros(d_model)
    x_prev=np.zeros(d_model)
    for i in range(seq_len):

    # 2. LayerNorm → Time Mixing
      out,num,den=time_mixing(layernorm(x[i]),layernorm(x_prev),num,den)
      out+=x[i]

    # 3. LayerNorm → Channel Mixing
      out+=channel_mixing(layernorm(x[i]),layernorm(x_prev))
      outputs.append(out)
      x_prev=x[i]

    output=np.array(outputs)
    return output

In [7]:
np.random.seed(42)
seq_len = 4
d_model = 16

x = np.random.randn(seq_len, d_model)
output = rwkv_block(x)
print("output shape:", output.shape)  # (4, 16)

output shape: (4, 16)
